# CottonLens: mevcut deneyde kısa GPU teşhisi

Bu defter `research-free-data-v1` deneyinin **hazır ve dondurulmuş** verisini kullanır. **Tümünü çalıştır** sekiz küçük sentetik GPU kontrolünü başlatır; geçmiş A turunu, ağır feature ablation veya geniş model aramasını başlatmaz. Bu kontroller piyasa tahmin başarısı değildir. Başarılı sonuçlar Drive’da kaydedilir ve aynı deney tekrar açılınca doğrulanarak yüklenir.

Yeni USDA/ECB kaynakları bu dondurulmuş deneye dahil değildir: gerçek yayın zamanı kanıtı tamamlanmadan tarihsel modele alınmayacaklar. Yanlış kaynak/deney kimliği veya etkin writer kilidi görülürse defter durur; kilidi elle silmeyin.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'research-free-data-v1'  # Retain this identifier for resumes of EXACTLY this data/source.


In [ ]:
import hashlib, importlib, json, sys, zipfile
from pathlib import PurePosixPath
SOURCE_ZIP = DRIVE_ROOT / 'sources/cottonlens-free-research-source-v1.zip'
EXPECTED_SOURCE_ID = 'c669ca9a9d4db2647b80993881efda2ba65b050ea26c6fd16a06a5c1e5dd1b93'
EXPECTED_ZIP_SHA256 = 'ff4eec811ef9549e7311a467475cca2b7826044566aecf5704b792668b06c6b1'
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert expected_zip == EXPECTED_ZIP_SHA256, 'Source ZIP sidecar differs from pinned bundle'
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_ZIP_SHA256, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'] == EXPECTED_SOURCE_ID, 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
PYTHON = setup(research=True)  # Locked managed Python 3.12.11; no global packages.
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
import json
from colab_setup import process_env

assert identity == EXPECTED_SOURCE_ID, 'Wrong source bundle; use the pinned free-research ZIP'
folder = DRIVE_ROOT / 'experiments' / EXPERIMENT
ready_path = folder / 'ready.json'
assert ready_path.is_file(), 'Prepared experiment missing; do not create a new experiment here'
ready = json.loads(ready_path.read_text(encoding='utf-8'))
assert ready['identity']['source_id'] == identity, 'Experiment and source bundle differ'
assert ready['identity']['profile'] == 'free-data-v1', 'Wrong experiment profile'
assert len(ready['identity']['split']['folds']) == 8, 'Expected eight frozen historical folds'
assert not ready['identity']['publication_sources'], 'Unexpected new data package: review before training'
assert not (folder / '.writer-lock').exists(), 'Writer lock present: verify old Colab session stopped before recovery'
BASE = [PYTHON, '-m', 'cottonlens_ml.research.engine', '--repo', REPO,
        '--drive-root', DRIVE_ROOT, '--experiment', EXPERIMENT]
run([*BASE, '--stage', 'status'])  # Read-only inventory; no model fit.
print('Pinned source and prepared cohort match. Next cell runs the eight small diagnostic fits.')


In [ ]:
import colab_progress
from IPython.display import display, Markdown

# Already completed diagnostics are displayed; they are not trained again.
diagnosis_path = folder / 'diagnosis.json'
if diagnosis_path.exists():
    prior = json.loads(diagnosis_path.read_text(encoding='utf-8'))
    assert prior['status'] == 'passed', 'Existing diagnosis failed; investigate it before any new experiment'
    print('Verified existing diagnosis loaded; no new fits requested.')
else:
    panel = display(Markdown('Starting eight small synthetic GPU controls.'), display_id=True)
    def update(message):
        panel.update(Markdown('```text\n' + message + '\n```'))
    colab_progress.supervise(
        [*BASE, '--stage', 'diagnose'], cwd=REPO, env=process_env(),
        drive_logs=DRIVE_ROOT / 'reports', decisions=folder / 'ledger' / 'completed',
        total=8, decision_pattern='*.json', update=update,
    )
    prior = json.loads(diagnosis_path.read_text(encoding='utf-8'))
    assert prior['status'] == 'passed', 'Diagnostic gate did not pass'
print('Diagnostic status:', prior['status'])
for name, result in prior['controls'].items():
    print(name, '| device:', result.get('device', 'recorded in fit'),
          '| synthetic relative MAE gain:', result.get('relative_mae_gain', 'smoke only'))
run([*BASE, '--stage', 'status'])
print('Next decision: inspect these controls before running the long feature ablation.')
